# 08 — Comparison with published Bengali meme models (DORA, MAF) under OCR noise
**Requires:** notebooks 03 and 04 finished (their predictions are loaded for the comparison). Kernel **Python (internship1)**.

Re-implements, from the authors' official code, the two published state-of-the-art models on our datasets:

| Model | Paper | Official code | Image | Text | Fusion |
|---|---|---|---|---|---|
| **DORA** | Hossain et al., ACL 2024 (BHM) | `eftekhar-hossain/Bengali-Hateful-Memes` | CLIP ViT-B/32 (frozen) | XGLM-564M | dual co-attention (2 heads) |
| **MAF** | Ahsan et al., EACL 2024 (MIMOSA) | `shawlyahsan/Bengali-Aggression-Memes` | CLIP ViT-B/32 (frozen) | Bangla-BERT (sagorsarker) | attentive fusion (16 heads) |

Each model is trained in two ways, 3 seeds each, on BHM, MIMOSA 5-class and MIMOSA binary:
* **train gold** — as published (gold captions);
* **mixed text** — uniform mixture of gold / Surya / EasyOCR text (same as our Fusion-mixed baseline), so they get the same robustness recipe as our baseline.

All models are tested on gold, Surya, EasyOCR and Surya+fallback (OCR) text with the same metrics as notebooks 04–05.

**Kept from the official code:** architecture, encoders, image preprocessing (224×224 resize + ImageNet normalization), max text length (DORA 50, MAF 70), attention heads, learning rate, linear decay schedule, unweighted cross-entropy.
**Changed to fit an 8 GB GPU / our protocol (reported in the paper):** batch 16, at most 5 epochs with early stopping (patience 2) on validation macro-F1 (instead of accuracy), frozen word-embedding matrix (as in all our robustness experiments), DORA trained with 8-bit AdamW instead of MADGRAD, binary tasks use a 2-way softmax instead of a sigmoid, CLIP image features pre-computed once (CLIP is frozen in both models, so this is identical).

**Runtime:** ~4 h (36 trainings), resumable. Lower `SEEDS` to `[42]` for a quick check.

## 1. Imports & config

In [ ]:
import os, re, gc, json, time, random, platform, unicodedata
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set before importing torch
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
import transformers, sklearn
from transformers import AutoTokenizer, AutoModel, CLIPVisionModelWithProjection, get_linear_schedule_with_warmup
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score
from scipy.stats import ttest_rel
from rapidfuzz.distance import Levenshtein
import matplotlib.pyplot as plt

ROOT = "/mnt/f415dcc8-27a5-4e66-8493-25f1d939974c/internshipwork/journal"
P, FEAT = f"{ROOT}/processed", f"{ROOT}/features"
NB03, NB04 = f"{ROOT}/results/03_ocr_eval", f"{ROOT}/results/04_nafim"
RES = f"{ROOT}/results/08_sota_comparison"
PRED = f"{RES}/preds"
os.makedirs(PRED, exist_ok=True)

TASKS = [("bhm", "label"), ("mimosa", "label"), ("mimosa", "label_bin")]
SEEDS = [42, 43, 44]
NAFIM_SEEDS = [42, 43, 44, 45, 46]          # notebook 04
REAL_SOURCES = {"gold": "text", "surya": "text_surya", "easyocr": "text_easyocr", "ocr": "text_ocr"}
EVAL_SOURCES = ["gold", "surya", "easyocr"]
HIGH_CER = 0.5
N_BOOT = 2000
BS, EVAL_BS = 16, 64
MAX_EPOCHS, PATIENCE = 5, 2

# official settings (architecture.py / models.py / main.py of each repository)
SPECS = {
    "DORA": dict(hf="facebook/xglm-564M", d=1024, max_len=50, heads=2, lr=2e-5, wd=0.0,
                 hidden=32, drop=0.01, dual=True, opt="adamw8bit"),
    "MAF":  dict(hf="sagorsarker/bangla-bert-base", d=768, max_len=70, heads=16, lr=5e-5, wd=0.01,
                 hidden=128, drop=0.2, dual=False, opt="adamw"),
}
REGIMES = ["train gold", "mixed text"]
MODELS = [f"{m} ({r})" for m in SPECS for r in REGIMES]

cfg04 = json.load(open(f"{NB04}/config.json"))
FREEZE_WORD_EMB = bool(cfg04["word_embeddings_frozen"])
MIXED_MIX = cfg04["mixed_baseline"]["text_source_mix"]
print("frozen word embeddings:", FREEZE_WORD_EMB, "| mixed-text mix:", MIXED_MIX)

dev = "cuda"
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| AMP:", AMP_DTYPE)
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 2**30:.2f} / {total / 2**30:.2f} GiB  (close other GPU programs if below ~6.5 GiB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def slug(s):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", s).strip("_")

## 2. Data, OCR error (for the high-CER subset)

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[^\u0980-\u09FFa-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def cer(ref, hyp):
    r, h = norm(ref), norm(hyp)
    return np.nan if not r else Levenshtein.distance(r, h) / len(r)

def idx_of(df, s):
    return np.where(df["split"].values == s)[0]

data = {}
for ds in sorted({d for d, _ in TASKS}):
    df = pd.read_csv(f"{P}/{ds}.csv").fillna("")
    for c in REAL_SOURCES.values():
        df[c] = df[c].astype(str)
    df["cer_ocr"] = [cer(r, h) for r, h in zip(df["text"], df["text_ocr"])]
    data[ds] = {"df": df}
    print(ds, df.shape, df["split"].value_counts().to_dict())

for ds, lab in TASKS:
    df = data[ds]["df"]
    y = df[lab].values[idx_of(df, "test")].astype(int)
    np.save(f"{PRED}/{ds}__{lab}__ytrue.npy", y)
    f04 = f"{NB04}/preds/{ds}__{lab}__ytrue.npy"
    if os.path.exists(f04):
        assert (np.load(f04) == y).all(), f"test order differs from notebook 04 for {ds}/{lab}"
print("test labels saved; order matches notebook 04")

## 3. CLIP ViT-B/32 image features with the official preprocessing (computed once, cached)

In [ ]:
# official dataset.py: Resize((224, 224)) + ToTensor + ImageNet normalization; architecture: frozen clip.visual (512-d projected output)
# HF CLIPVisionModelWithProjection.image_embeds == OpenAI clip visual output (ln_post(CLS) @ proj)
TF = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def clip_b32_features(ds):
    f = f"{FEAT}/{ds}_clipb32_official.npy"
    if os.path.exists(f):
        return np.load(f)
    df = data[ds]["df"]
    clip = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-base-patch32").to(dev).eval()
    out = []
    with torch.no_grad():
        for i in tqdm(range(0, len(df), 64), desc=f"CLIP B/32 {ds}"):
            x = torch.stack([TF(Image.open(p).convert("RGB")) for p in df["image_path"].iloc[i:i + 64]]).to(dev)
            out.append(clip(pixel_values=x).image_embeds.float().cpu().numpy())
    del clip; gc.collect(); torch.cuda.empty_cache()
    feats = np.concatenate(out).astype(np.float32)
    np.save(f, feats)
    return feats

for ds in data:
    data[ds]["img"] = torch.from_numpy(clip_b32_features(ds))
    assert len(data[ds]["img"]) == len(data[ds]["df"])
    print(ds, "image features:", tuple(data[ds]["img"].shape))

## 4. Tokenization (per model, per text source; padded to the official max length)

In [ ]:
TOKS = {}
def tokens(model_key, ds):
    """returns ids, mask of shape [n_sources, N, max_len] in the order of REAL_SOURCES"""
    k = (model_key, ds)
    if k not in TOKS:
        spec = SPECS[model_key]
        tok = AutoTokenizer.from_pretrained(spec["hf"])
        df = data[ds]["df"]
        enc = [tok(df[c].tolist(), padding="max_length", truncation=True, max_length=spec["max_len"], return_tensors="pt")
               for c in REAL_SOURCES.values()]
        TOKS[k] = (torch.stack([e["input_ids"] for e in enc]), torch.stack([e["attention_mask"] for e in enc]))
    return TOKS[k]

SRC_IDX = {s: i for i, s in enumerate(REAL_SOURCES)}
print(SRC_IDX)

## 5. DORA / MAF (faithful to the official architecture code)

In [ ]:
class CoAttentionFusion(nn.Module):
    """DORA (dual=True): concat[attn(q=img,k=txt,v=txt), attn(q=img,k=txt,v=img), img, txt]
       MAF  (dual=False): concat[attn(q=img,k=txt,v=img), img, txt]
       The 512-d CLIP vector is projected, repeated to the text length (adaptive average pooling of a length-1
       sequence, as in the original code), fused, mean-pooled over positions and classified by a 2-layer MLP."""
    def __init__(self, spec, n_classes):
        super().__init__()
        self.spec, self.L, self.dual = spec, spec["max_len"], spec["dual"]
        self.text = AutoModel.from_pretrained(spec["hf"])
        if FREEZE_WORD_EMB:
            self.text.get_input_embeddings().weight.requires_grad = False
        d = spec["d"]
        self.visual_linear = nn.Linear(512, d)
        self.attention = nn.MultiheadAttention(d, spec["heads"], dropout=0.1)   # sequence-first, one shared module
        self.fc = nn.Sequential(nn.Linear((4 if self.dual else 3) * d, spec["hidden"]), nn.ReLU(),
                                nn.Dropout(spec["drop"]), nn.Linear(spec["hidden"], n_classes))

    def forward(self, ids, mask, img):
        v = self.visual_linear(img).unsqueeze(1)
        v = F.adaptive_avg_pool1d(v.permute(0, 2, 1), self.L).permute(0, 2, 1)       # [B, L, d]
        t = self.text(input_ids=ids, attention_mask=mask).last_hidden_state            # [B, L, d]
        vq, tk = v.permute(1, 0, 2), t.permute(1, 0, 2)
        a_img = self.attention(vq, tk, vq)[0].permute(1, 0, 2)
        parts = [self.attention(vq, tk, tk)[0].permute(1, 0, 2), a_img] if self.dual else [a_img]
        z = torch.cat(parts + [v, t], dim=2)
        return self.fc(z.mean(1))

def make_optimizer(model, spec):
    params = [p for p in model.parameters() if p.requires_grad]
    if spec["opt"] == "adamw8bit":
        try:
            import bitsandbytes as bnb
            return bnb.optim.AdamW8bit(params, lr=spec["lr"], weight_decay=spec["wd"]), "AdamW8bit"
        except Exception as e:
            print("bitsandbytes unavailable -> torch AdamW:", str(e)[:80])
    return torch.optim.AdamW(params, lr=spec["lr"], weight_decay=spec["wd"], foreach=False), "AdamW"

## 6. Training (resumable) — best epoch chosen on validation macro-F1

In [ ]:
@torch.no_grad()
def predict(model, model_key, ds, src, idx):
    ids, mask = tokens(model_key, ds)
    img = data[ds]["img"]
    model.eval(); out = []
    for i in range(0, len(idx), EVAL_BS):
        b = torch.as_tensor(idx[i:i + EVAL_BS])
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(ids[SRC_IDX[src], b].to(dev), mask[SRC_IDX[src], b].to(dev), img[b].to(dev))
        out.append(logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(out)

def run_train(name, ds, lab, seed, ckpt=False):
    model_key, regime = name.split(" (")[0], name.split(" (")[1].rstrip(")")
    spec = SPECS[model_key]
    set_seed(seed)
    df = data[ds]["df"]; y_all = df[lab].values.astype(int); n = int(y_all.max()) + 1
    tr, va, te = idx_of(df, "train"), idx_of(df, "val"), idx_of(df, "test")
    ids, mask = tokens(model_key, ds); img = data[ds]["img"]
    model = CoAttentionFusion(spec, n).to(dev)
    if ckpt:
        model.text.gradient_checkpointing_enable()
    opt, opt_name = make_optimizer(model, spec)
    steps = MAX_EPOCHS * int(np.ceil(len(tr) / BS))
    sched = get_linear_schedule_with_warmup(opt, 0, steps)            # official: no warm-up, linear decay
    g = np.random.default_rng(seed)
    mix_src = [s for s in MIXED_MIX if MIXED_MIX[s] > 0]
    mix_p = np.array([MIXED_MIX[s] for s in mix_src]); mix_p = mix_p / mix_p.sum()
    val_sources = ["gold"] if regime == "train gold" else EVAL_SOURCES
    best, bad, best_out, best_ep = -1, 0, None, 0
    t0 = time.time()
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = g.permutation(tr)
        for i in range(0, len(perm), BS):
            b = torch.as_tensor(perm[i:i + BS])
            if regime == "train gold":
                s_idx = torch.zeros(len(b), dtype=torch.long)
            else:
                s_idx = torch.as_tensor([SRC_IDX[s] for s in g.choice(mix_src, size=len(b), p=mix_p)])
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(ids[s_idx, b].to(dev), mask[s_idx, b].to(dev), img[b].to(dev))
            loss = F.cross_entropy(logits.float(), torch.as_tensor(y_all[b.numpy()]).to(dev))   # unweighted, as published
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        f = np.mean([f1_score(y_all[va], predict(model, model_key, ds, s, va).argmax(1), average="macro") for s in val_sources])
        print(f"  epoch {ep + 1}: val macro-F1 ({'/'.join(val_sources)}) = {f:.4f}")
        if f > best:
            best, bad, best_ep = f, 0, ep + 1
            best_out = {s: predict(model, model_key, ds, s, te) for s in REAL_SOURCES}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return best_out, best, best_ep, (time.time() - t0) / 60, opt_name

RUNS_CSV = f"{RES}/runs.csv"
done = set(pd.read_csv(RUNS_CSV)["train_id"]) if os.path.exists(RUNS_CSV) else set()
jobs = [(m, ds, lab, s) for s in SEEDS for m in MODELS for ds, lab in TASKS]
print("trainings already done:", len(done & {f"{d}__{l}__{slug(m)}__s{s}" for m, d, l, s in jobs}), "of", len(jobs))

for name, ds, lab, seed in jobs:
    train_id = f"{ds}__{lab}__{slug(name)}__s{seed}"
    if train_id in done:
        continue
    print(f"\n=== {train_id}")
    try:
        out, vbest, ep, mins, opt_name = run_train(name, ds, lab, seed)
    except torch.cuda.OutOfMemoryError:
        print("CUDA out of memory -> retrying with gradient checkpointing")
        gc.collect(); torch.cuda.empty_cache()
        out, vbest, ep, mins, opt_name = run_train(name, ds, lab, seed, ckpt=True)
    y = np.load(f"{PRED}/{ds}__{lab}__ytrue.npy")
    for s, p in out.items():
        np.save(f"{PRED}/{train_id}__test-{s}.npy", p)
    row = {"train_id": train_id, "dataset": ds, "task": lab, "model": name, "seed": seed, "val_f1": vbest,
           "best_epoch": ep, "optimizer": opt_name, "minutes": mins,
           **{f"test_{s}": f1_score(y, p.argmax(1), average="macro") for s, p in out.items()}}
    pd.DataFrame([row]).to_csv(RUNS_CSV, mode="a", header=not os.path.exists(RUNS_CSV), index=False)
    print(f"{name} | {ds}/{lab} | seed {seed}: " + "  ".join(f"{s}={row['test_' + s]:.4f}" for s in REAL_SOURCES) + f"  ({mins:.1f} min)")
print("\nall trainings done")

## 7. Gather predictions: DORA / MAF (this notebook) + our fusion baselines and NAFIM (notebooks 03–04)

In [ ]:
PRED03, PRED04 = f"{NB03}/preds", f"{NB04}/preds"
cfg03 = json.load(open(f"{NB03}/config.json"))
BASE_SLUG = cfg03["selected_model"].replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")
def nb04_slug(s):
    return s.replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")

def pred_path(model, ds, lab, s, t):
    if model == "Fusion (train gold)":
        return f"{PRED03}/{ds}__{lab}__{BASE_SLUG}__train-gold__s{s}__test-{t}.npy"
    if model == "Fusion (train OCR)":
        return f"{PRED03}/{ds}__{lab}__{BASE_SLUG}__train-ocr__s{s}__test-{t}.npy"
    if model in ("Fusion (mixed text)", "NAFIM"):
        return f"{PRED04}/{ds}__{lab}__{nb04_slug(model)}__s{s}__test-{t}.npy"
    return f"{PRED}/{ds}__{lab}__{slug(model)}__s{s}__test-{t}.npy"

def load_preds(model, ds, lab, seeds):
    out = {}
    for s in seeds:
        files = {t: pred_path(model, ds, lab, s, t) for t in REAL_SOURCES}
        if all(os.path.exists(f) for f in files.values()):
            out[s] = {t: np.load(f).argmax(1) for t, f in files.items()}
    return out

def hi_mask(ds):
    df = data[ds]["df"]
    return df["cer_ocr"].values[idx_of(df, "test")] >= HIGH_CER

METRICS = ["gold", "surya", "easyocr", "ocr", "mean", "worst", "high-CER"]
def metric_rows(model, ds, lab, seeds):
    y = np.load(f"{PRED}/{ds}__{lab}__ytrue.npy"); hi = hi_mask(ds); rows = []
    for s, pr in load_preds(model, ds, lab, seeds).items():
        f1 = {t: f1_score(y, pr[t], average="macro") for t in REAL_SOURCES}
        ev = [f1[t] for t in EVAL_SOURCES]
        rows.append({"dataset": ds, "task": lab, "model": model, "seed": s, **f1, "mean": np.mean(ev),
                     "worst": np.min(ev), "high-CER": f1_score(y[hi], pr["ocr"][hi], average="macro") if hi.any() else np.nan})
    return rows

def fmt(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else (f"{x.mean():.4f}" if len(x) else "—")

OURS = ["Fusion (train gold)", "Fusion (train OCR)", "Fusion (mixed text)", "NAFIM"]
ALL_MODELS = MODELS + OURS
rows = [r for ds, lab in TASKS for m in MODELS for r in metric_rows(m, ds, lab, SEEDS)]
rows += [r for ds, lab in TASKS for m in OURS for r in metric_rows(m, ds, lab, NAFIM_SEEDS)]
cmp = pd.DataFrame(rows)
print("seeds per model:", cmp.groupby("model")["seed"].nunique().to_dict())
table = (cmp.groupby(["dataset", "task", "model"])[METRICS].agg(fmt)
         .reindex(pd.MultiIndex.from_tuples([(d, t, m) for d, t in TASKS for m in ALL_MODELS]), fill_value="—"))
table

## 8. Do the published models degrade on OCR text? (gold-trained, gold test vs. OCR test; paired t-test over seeds)

In [ ]:
deg = []
for ds, lab in TASKS:
    for m in [f"{k} (train gold)" for k in SPECS] + ["Fusion (train gold)"]:
        sub = cmp[(cmp.dataset == ds) & (cmp.task == lab) & (cmp.model == m)].sort_values("seed")
        if len(sub) < 2:
            continue
        for t in ["surya", "easyocr", "ocr"]:
            p = ttest_rel(sub["gold"], sub[t]).pvalue
            deg.append({"dataset": ds, "task": lab, "model": m, "test": t, "gold_f1": sub["gold"].mean(),
                        "test_f1": sub[t].mean(), "delta_points": 100 * (sub[t] - sub["gold"]).mean(),
                        "p_value": p, "n_seeds": len(sub)})
deg_df = pd.DataFrame(deg)
deg_df.pivot_table(index=["dataset", "task", "model"], columns="test", values="delta_points").round(2)

## 9. Paired bootstrap: NAFIM vs. DORA / MAF (test memes resampled, common seeds 42–44)

In [ ]:
def fast_macro_f1(y, p, n):
    cm = np.bincount(n * y + p, minlength=n * n).reshape(n, n)
    tp = np.diag(cm).astype(float); pred = cm.sum(0); true = cm.sum(1)
    present = (pred + true) > 0
    f1 = np.where(present, 2 * tp / np.maximum(pred + true, 1), 0.0)
    return f1[present].mean()

def boot_metric(pr, y, idx, n, kind, hi_idx=None):
    vals = []
    for s, d in pr.items():
        if kind == "high":
            vals.append(fast_macro_f1(y[hi_idx], d["ocr"][hi_idx], n)); continue
        f = [fast_macro_f1(y[idx], d[t][idx], n) for t in EVAL_SOURCES]
        vals.append(np.mean(f) if kind == "mean" else np.min(f))
    return np.mean(vals)

boot = []
rng = np.random.default_rng(0)
for ds, lab in TASKS:
    y = np.load(f"{PRED}/{ds}__{lab}__ytrue.npy"); n = int(y.max()) + 1; N = len(y)
    hi = np.where(hi_mask(ds))[0]
    A = load_preds("NAFIM", ds, lab, SEEDS)
    for base in MODELS:
        B = load_preds(base, ds, lab, SEEDS)
        common = sorted(set(A) & set(B))
        if len(common) < 2:
            continue
        a = {s: A[s] for s in common}; b = {s: B[s] for s in common}
        for kind in (["mean", "worst", "high"] if len(hi) else ["mean", "worst"]):
            full = np.arange(N)
            obs = boot_metric(a, y, full, n, kind, hi) - boot_metric(b, y, full, n, kind, hi)
            diffs = np.empty(N_BOOT)
            for k in range(N_BOOT):
                idx = rng.integers(0, N, N); hidx = hi[rng.integers(0, len(hi), len(hi))]
                diffs[k] = boot_metric(a, y, idx, n, kind, hidx) - boot_metric(b, y, idx, n, kind, hidx)
            p = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
            boot.append({"dataset": ds, "task": lab, "vs": base, "metric": kind, "delta_points": 100 * obs,
                         "ci95_low": 100 * np.percentile(diffs, 2.5), "ci95_high": 100 * np.percentile(diffs, 97.5),
                         "p_value": p, "n_seeds": len(common)})
boot_df = pd.DataFrame(boot)
boot_df["result"] = np.where(boot_df["p_value"] < 0.05, np.where(boot_df["delta_points"] > 0, "NAFIM better", "NAFIM worse"), "n.s.")
display(boot_df.pivot_table(index=["dataset", "task", "vs"], columns="metric", values="delta_points", aggfunc="first").round(2))
boot_df.pivot_table(index=["dataset", "task", "vs"], columns="metric", values="result", aggfunc="first")

## 10. Figure: gold vs. OCR macro-F1 of every model (mean over seeds)

In [ ]:
FIG_MODELS = MODELS + OURS
fig, axes = plt.subplots(1, len(TASKS), figsize=(4.6 * len(TASKS), 3.8))
for ax, (ds, lab) in zip(axes, TASKS):
    sub = cmp[(cmp.dataset == ds) & (cmp.task == lab)].groupby("model")[["gold", "ocr", "worst"]].mean().reindex(FIG_MODELS)
    x = np.arange(len(sub)); w = 0.27
    for k, (col, lbl) in enumerate([("gold", "gold text"), ("ocr", "OCR text"), ("worst", "worst source")]):
        ax.bar(x + (k - 1) * w, sub[col], w, label=lbl)
    lo = np.nanmin(sub[["gold", "ocr", "worst"]].values)
    ax.set_ylim(max(0, lo - 0.05), min(1, np.nanmax(sub[["gold", "ocr", "worst"]].values) + 0.03))
    ax.set_xticks(x); ax.set_xticklabels(sub.index, rotation=60, ha="right", fontsize=7)
    ax.set_title(f"{ds} / {lab}", fontsize=9); ax.set_ylabel("macro-F1")
axes[0].legend(fontsize=7, loc="lower left")
fig.tight_layout(); fig.savefig(f"{RES}/sota_gold_vs_ocr.png", dpi=300, bbox_inches="tight"); plt.show()

## 11. Save results + config

In [ ]:
cmp.to_csv(f"{RES}/A_per_seed.csv", index=False)
table.to_csv(f"{RES}/A_table.csv")
deg_df.to_csv(f"{RES}/B_degradation_gold_trained.csv", index=False)
boot_df.to_csv(f"{RES}/C_bootstrap_nafim_vs_sota.csv", index=False)
config = {
    "models": SPECS, "regimes": REGIMES, "tasks": TASKS, "seeds": SEEDS, "nafim_seeds": NAFIM_SEEDS,
    "batch_size": BS, "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "selection": "validation macro-F1 (gold for train-gold; mean over gold/surya/easyocr for mixed)",
    "image_features": "CLIP ViT-B/32 (openai/clip-vit-base-patch32) image_embeds, Resize 224x224 + ImageNet normalization (official preprocessing), frozen",
    "word_embeddings_frozen": FREEZE_WORD_EMB, "mixed_text_mix": MIXED_MIX, "loss": "unweighted cross-entropy (as published)",
    "deviations_from_official": ["batch 16", "max 5 epochs + early stopping on val macro-F1", "frozen word embeddings",
                                 "DORA: 8-bit AdamW instead of MADGRAD", "2-way softmax instead of sigmoid for binary tasks",
                                 "pre-computed frozen CLIP features"],
    "bootstrap": {"n": N_BOOT, "seeds": "common seeds of NAFIM and baseline"},
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
                 "sklearn": sklearn.__version__, "gpu": torch.cuda.get_device_name(0)},
}
with open(f"{RES}/config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("saved to", RES)